# Modern Python development with uv
## 1. Setting up a development environment
### 1.1 Depsight Dependency Manager

In [43]:
!git clone https://github.com/ValentinTwin1206/depsight-dependency-manager.git

Cloning into 'depsight-dependency-manager'...
remote: Enumerating objects: 627, done.
remote: Counting objects: 100% (229/229), done.
remote: Compressing objects: 100% (126/126), done.
remote: Total 627 (delta 108), reused 149 (delta 89), pack-reused 398 (from 1)
Receiving objects: 100% (627/627), 2.50 MiB | 5.92 MiB/s, done.
Resolving deltas: 100% (299/299), done.


In [44]:
# once the repository is cloned, we need to install the dependencies and setup the environment
!cd depsight-dependency-manager && uv sync

Using CPython 3.12.3 interpreter at: /usr/bin/python3.12
Creating virtual environment at: .venv
Resolved 25 packages in 20ms                                         
Prepared 1 package in 73ms                                               
Installed 24 packages in 121ms                              
 + build==1.4.2
 + click==8.3.1
 + depsight==1.3.0 (from file:///home/fixcfhu/repos/ValentinTwin1206/modern-python-devops-egineering/projects/proj8_depsight/depsight-dependency-manager)
 + iniconfig==2.3.0
 + librt==0.8.1
 + linkify-it-py==2.1.0
 + markdown-it-py==4.0.0
 + mdit-py-plugins==0.5.0
 + mdurl==0.1.2
 + mypy==1.19.1
 + mypy-extensions==1.1.0
 + packaging==26.0
 + pathspec==1.0.4
 + platformdirs==4.9.4
 + pluggy==1.6.0
 + pygments==2.19.2
 + pyproject-hooks==1.2.0
 + pytest==9.0.2
 + rich==14.3.3
 + rich-click==1.9.7
 + ruff==0.15.8
 + textual==8.2.1
 + typing-extensions==4.15.0
 + uc-micro-py==2.0.0


In [45]:
# let's see if depsight is running as expected
!cd depsight-dependency-manager && uv run depsight --help

                                                                                
   ____                 _       _     _                                         
  |  _ \  ___ _ __  ___(_) __ _| |__ | |_                                       
  | | | |/ _ \ '_ \/ __| |/ _` | '_ \| __|                                      
  | |_| |  __/ |_) \__ \ | (_| | | | | |_                                       
  |____/ \___| .__/|___/_|\__, |_| |_|\__|                                      
             |_|          |___/                                                 
  A modern dependency analysis CLI                                              
                                                                                
 Usage: depsight [OPTIONS] COMMAND [ARGS]...                                    
                                                                                
 A modern TUI framework for scanning local project dependencies.                
                            

### 1.2 Python's Plugin Discovery Concept

According to the documentation, depsight provides a plugin mechanism that allows third-party plugins to extend its functionality. The built-in plugins are registered via the `pyproject.toml` entry points:

```toml
# Plugin support
[project.entry-points."depsight.plugins"]
uv = "depsight.core.plugins.uv.uv:UVPlugin"
vsce = "depsight.core.plugins.vsce.vsce:VSCEPlugin"
```

These are the plugins that `depsight` ships with by default. However, the same entry point mechanism is also used to discover and load additional third-party plugins.

A first indication of this behavior can be found in ``cli.py``, where all supported plugins are registered during startup:

```python
#
# PLUGIN REGISTRATION
# # # # # # # # #
for _name in SUPPORTED_PLUGINS:
    _register_plugin(_name)
```

Following the registration flow eventually leads to the ``discover_plugin_files()`` function in ``utils.py``, which instantiates every registered plugin and extracts its metadata:

```python
for name, plugin_cls in plugin_registry.items():
        try:
            instance = plugin_cls()
            registry[name] = (instance.dependency_files, instance.default_file)
        except Exception:
            raise SystemExit(f"Failed to inspect plugin '{name}'.")
```

In short, any package that exposes a `depsight.plugins` entry point can be discovered and loaded by `depsight`. This enables third-party plugins to integrate seamlessly without requiring any changes to the core application.

### 1.3 Installing dependencies as editable

Based on the research our goal is to implement a third-party-plugin for the package manager `poetry` which itself is a very famous 
package manager for Python based applications. 

In [46]:
# clone the depsight poetry plugin
!git clone https://github.com/Pravi1206/depsight-poetry-plugin.git

Cloning into 'depsight-poetry-plugin'...
remote: Enumerating objects: 125, done.
remote: Counting objects: 100% (125/125), done.
remote: Compressing objects: 100% (100/100), done.
remote: Total 125 (delta 13), reused 114 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (125/125), 1.59 MiB | 5.36 MiB/s, done.
Resolving deltas: 100% (13/13), done.


The project is derived from the depsight-third-party-plugin template and it already offers a ``entrypoint`` integration for `depsight` - let's check the `pyproject.toml`, which shows us:

```toml
[project.entry-points."depsight.plugins"]
myplugin = "poetryplugin.poetryplugin:PoetryPlugin"
```

To test the ``poetry`` plugin, we need to install it in the virtual environment as a ``editable`` dependency.

In [47]:
!cd depsight-dependency-manager && uv add --editable ../depsight-poetry-plugin && uv run depsight --help

Resolved 26 packages in 24ms                                         
Prepared 2 packages in 324ms                                             
Uninstalled 1 package in 0.83ms
Installed 2 packages in 2msin==0.1.0 (from file:///home/fixc
 ~ depsight==1.3.0 (from file:///home/fixcfhu/repos/ValentinTwin1206/modern-python-devops-egineering/projects/proj8_depsight/depsight-dependency-manager)
 + depsight-poetry-plugin==0.1.0 (from file:///home/fixcfhu/repos/ValentinTwin1206/modern-python-devops-egineering/projects/proj8_depsight/depsight-poetry-plugin)
                                                                                
   ____                 _       _     _                                         
  |  _ \  ___ _ __  ___(_) __ _| |__ | |_                                       
  | | | |/ _ \ '_ \/ __| |/ _` | '_ \| __|                                      
  | |_| |  __/ |_) \__ \ | (_| | | | | |_                                       
  |____/ \___| .__/|___/_|\__, |_| |_|\__

However, installing the ``poetry`` plugin via the ``uv add`` command has the drawback, that it is going to be listed inside the ``pyproject.toml`` file of ``depsight`` itself. 

### 1.4 Setting up a workspace with uv

The `uv add` updates the `pyproject.toml` and sets the ``depsight-poetry-plugin`` as a dependency for `depsight` itself. Moreover, having both projects loosly coupled from each other increases the effort for a clean dependency-management. 
Having an approach which supports the co-existence of both projects might be very handy for development. Thanks to `uv` we can use the concept of [workspaces](https://docs.astral.sh/uv/concepts/projects/workspaces/) offered by ``uv`` itself.
The benefit of an **workspace** is a shared, global ``uv.lock`` that is enforced over the whole workspace. Moreover, the ``members`` in a workspace remain independent but are seamlessly integrated.

In [48]:
# we are going to undo the previous changes and remove the depsight-poetry-plugin from the pyproject.toml
!cd depsight-dependency-manager && uv remove depsight-poetry-plugin

Resolved 25 packages in 30ms                                         
Prepared 1 package in 68ms                                               
Uninstalled 2 packages in 1ms
Installed 1 package in 3msom file:///home/fixcfhu/repos/Vale
 ~ depsight==1.3.0 (from file:///home/fixcfhu/repos/ValentinTwin1206/modern-python-devops-egineering/projects/proj8_depsight/depsight-dependency-manager)
 - depsight-poetry-plugin==0.1.0 (from file:///home/fixcfhu/repos/ValentinTwin1206/modern-python-devops-egineering/projects/proj8_depsight/depsight-poetry-plugin)


In [49]:
# we are creating an uv workspace for both projects
!mkdir depsight && mv depsight-dependency-manager depsight && mv depsight-poetry-plugin depsight

Now we need to create a workspace root ``pyproject.toml`` file in the current directory (where ``uv_workspace/`` was created). This file will configure the workspace structure:

```toml
[tool.uv.workspace]
members = [
    "depsight-dependency-manager",
    "depsight-poetry-plugin",
]

[dependency-groups]
dev = [
    "pytest-cov>=7.1.0",
]
```

Additionally, workspace members that depend on each other should declare that dependency via ``tool.uv.sources``. For the poetry plugin to depend on depsight, add this to ``uv_workspace/depsight-poetry-plugin/pyproject.toml``:

```toml
[tool.uv.sources]
depsight = { workspace = true }
```

In [52]:
!cd depsight && uv sync && uv run depsight --help

Resolved 58 packages in 215ms                                        
Prepared 2 packages in 327ms                                             
Installed 28 packages in 67ms                               
 + ast-serialize==0.6.0
 + build==1.5.0
 + click==8.4.2
 + coverage==7.15.0
 + depsight==1.3.0 (from file:///home/fixcfhu/repos/ValentinTwin1206/modern-python-devops-egineering/projects/proj8_depsight/depsight/depsight-dependency-manager)
 + depsight-poetry-plugin==0.1.0 (from file:///home/fixcfhu/repos/ValentinTwin1206/modern-python-devops-egineering/projects/proj8_depsight/depsight/depsight-poetry-plugin)
 + iniconfig==2.3.0
 + librt==0.12.0
 + linkify-it-py==2.1.0
 + markdown-it-py==4.2.0
 + mdit-py-plugins==0.6.1
 + mdurl==0.1.2
 + mypy==2.2.0
 + mypy-extensions==1.1.0
 + packaging==26.2
 + pathspec==1.1.1
 + platformdirs==4.10.0
 + pluggy==1.6.0
 + pygments==2.20.0
 + pyproject-hooks==1.2.0
 + pytest==9.1.1
 + pytest-cov==7.1.0
 + rich==15.0.0
 + rich-click==1.9.8
 + ruff==0.15.2

---

## 2. Implementation of a poetry plugin for depsight
### 2.1 Installing poetry with uv's tool feature

Our ``poetry`` plugin for depsight should capture all dependencies in a poetry project. To implement this, we need to know how ``poetry`` works and we need to set up an example project which we can use as a base for our implementation. 
Before we are going to set up an example project, we need to install ``poetry`` on our system. In the previous slides we already heared about the interface that ``uv`` provides for ``tools`` which are Python packages that offer a cli to the user. To install a tool you can simply use the following command:

```bash
uv tool install poetry
```

This will install ``poetry`` and its dependencies in a temporary virtual environment isolated from the current project.

TODO: research about tools a bit more and if they can be deleted with a cache prune

In [61]:
!cd depsight && uv tool install poetry --verbose

DEBUG uv 0.8.18
DEBUG Searching for default Python interpreter in managed installations or search path
DEBUG Searching for managed installations at `/home/fixcfhu/.local/share/uv/python`
DEBUG Found `cpython-3.12.3-linux-x86_64-gnu` at `/home/fixcfhu/repos/ValentinTwin1206/modern-python-devops-egineering/.venv/bin/python` (first executable in the search path)
DEBUG Ignoring Python interpreter at `/home/fixcfhu/repos/ValentinTwin1206/modern-python-devops-egineering/.venv/bin/python`: system interpreter required
DEBUG Found `cpython-3.12.3-linux-x86_64-gnu` at `/home/fixcfhu/repos/ValentinTwin1206/modern-python-devops-egineering/.venv/bin/python3` (search path)
DEBUG Ignoring Python interpreter at `/home/fixcfhu/repos/ValentinTwin1206/modern-python-devops-egineering/.venv/bin/python3`: system interpreter required
DEBUG Found `cpython-3.12.3-linux-x86_64-gnu` at `/home/fixcfhu/repos/ValentinTwin1206/modern-python-devops-egineering/.venv/bin/python3.12` (search path)
DEBUG Ignoring Python 

In [7]:
ls -la ~/.local/share/uv/tools

total 28
drwxr-xr-x 6 fixcfhu fixcfhu 4096 Jul  8 08:10 ./
drwxr-xr-x 5 fixcfhu fixcfhu 4096 Jul  2 07:21 ../
-rw-r--r-- 1 fixcfhu fixcfhu    1 Jan 21 12:57 .gitignore
-rwxrwxrwx 1 fixcfhu fixcfhu    0 Jan 21 12:57 .lock*
drwxr-xr-x 4 fixcfhu fixcfhu 4096 Jun 29 15:02 black/
drwxr-xr-x 4 fixcfhu fixcfhu 4096 Jul  8 08:10 poetry/
drwxr-xr-x 4 fixcfhu fixcfhu 4096 Jun 29 15:00 ruff/
drwxr-xr-x 4 fixcfhu fixcfhu 4096 Jan 21 12:58 specify-cli/


In [5]:
uv tool list

/home/fixcfhu/repos/ValentinTwin1206/modern-python-devops-egineering/.venv/bin/python3: No module named uv
Note: you may need to restart the kernel to use updated packages.


In [66]:
!uv tool run poetry --version

Poetry (version 2.4.1)


### 2.2 Setting up an example poetry project

Since we have set up ``poetry`` with the tool feature of ``uv`` we can now move forward and set up an demo poject. We are going to follow the instruction from ``poetry's`` [documentation](https://python-poetry.org/docs/basic-usage/).

In [9]:
!cd depsight && uv tool run poetry new poetry-demo

Created package poetry_demo in poetry-demo


In [10]:
ls -la  depsight/poetry-demo

total 20
drwxr-xr-x 4 fixcfhu fixcfhu 4096 Jul  9 07:22 ./
drwxr-xr-x 6 fixcfhu fixcfhu 4096 Jul  9 07:22 ../
-rw-r--r-- 1 fixcfhu fixcfhu    0 Jul  9 07:22 README.md
-rw-r--r-- 1 fixcfhu fixcfhu  382 Jul  9 07:22 pyproject.toml
drwxr-xr-x 3 fixcfhu fixcfhu 4096 Jul  9 07:22 src/
drwxr-xr-x 2 fixcfhu fixcfhu 4096 Jul  9 07:22 tests/


In [11]:
!cd depsight/poetry-demo && uv tool run poetry add pendulum --verbose

Using virtualenv: /home/fixcfhu/repos/ValentinTwin1206/modern-python-devops-egineering/.venv
Checking keyring availability: Unavailable
Using version ^3.2.0 for pendulum

Updating dependencies
Resolving dependencies... (0.2s)

Finding the necessary packages for the current system

Package operations: 1 install, 0 updates, 0 removals, 3 skipped

  - Installing pendulum (3.2.0): Pending...
  - Installing python-dateutil (2.9.0.post0): Pending...
  - Installing six (1.17.0): Pending...
  - Installing six (1.17.0): Skipped for the following reason: Already installed
  - Installing six (1.17.0): Skipped for the following reason: Already installed
  - Installing python-dateutil (2.9.0.post0): Skipped for the following reason: Already installed
  - Installing six (1.17.0): Skipped for the following reason: Already installed
  - Installing tzdata (2026.2): Pending...
  - Installing tzdata (2026.2): Skipped for the following reason: Already installed
  - Installing python-dateutil (2.9.0.post0)

In [12]:
!cd depsight/poetry-demo && uv tool run poetry add pytest --group dev --verbose

Using virtualenv: /home/fixcfhu/repos/ValentinTwin1206/modern-python-devops-egineering/.venv
Checking keyring availability: Unavailable
Using version ^9.1.1 for pytest

Updating dependencies
Resolving dependencies... (0.5s)

Finding the necessary packages for the current system

Package operations: 3 installs, 0 updates, 0 removals, 6 skipped

  - Installing iniconfig (2.3.0): Pending...
  - Installing pluggy (1.6.0): Pending...
  - Installing pluggy (1.6.0): Pending...
  - Installing iniconfig (2.3.0): Downloading... 0%
  - Installing pluggy (1.6.0): Pending...
  - Installing pluggy (1.6.0): Pending...
  - Installing iniconfig (2.3.0): Downloading... 100%
  - Installing pluggy (1.6.0): Pending...
  - Installing pluggy (1.6.0): Downloading... 0%
  - Installing pluggy (1.6.0): Downloading... 0%
  - Installing iniconfig (2.3.0): Installing...
  - Installing pluggy (1.6.0): Downloading... 0%
  - Installing pluggy (1.6.0): Downloading... 0%
  - Installing iniconfig (2.3.0)
  - Installing p

### 2.3 Implementing the depsight plugin for poetry
#### 2.3.1 Understanding the `Dependency` interface

We have seen how easy it is to setup and faciliate a tool with the tool interface of ``uv``. For our implemention for ``poetry`` and the used dependencies within a project we need to take a deeper look in the ``poetry.lock`` file which is the central piece for ``poertry``'s dependency management. 

The `poetry.lock` file defines packages in `[[package]]` sections and specifies their dependencies in corresponding `[[package.dependencies]]` sections.

To support the `depsight` interface, we need to distinguish between direct and transitive dependencies. 

```python
class Dependency:
    name: str
    version: str | None = None
    constraint: str | None = None
    tool_name: str | None = None
    registry: str | None = None
    file: str | None = None
    category: packageType = "prod"
    is_transitive: bool = False
```

Packages listed as dependencies within a `[[package.dependencies]]` section are considered **transitive dependencies**, as they are required by another package. In contrast, top-level packages such as `pendulum` or `pytest` are treated as **direct (non-transitive) dependencies**, since they are explicitly declared by the project.

#### 2.3.2 Loading the ``poetry.lock`` file

At first, we need to load the ``poetry.lock`` file which can simply be done with the ``tomlib`` extension.

```python
@staticmethod
    def _load_dependency_files(project_dir: Path, filename: str) -> tuple[dict, Path] | None:
        """Walk *project_dir* for *filename*, parse the first match as TOML.

        Checks the project root first, then walks subdirectories.
        Returns the parsed data and the resolved file path, or
        `None` if no matching file is found.
        """
        root_candidate = project_dir / filename

        if root_candidate.is_file():
            with root_candidate.open("rb") as f:
                return tomllib.load(f), root_candidate.resolve()

        for path in project_dir.rglob(filename):
            if path.is_file():
                with path.open("rb") as f:
                    return tomllib.load(f), path.resolve()

        return None
```

This can easily be invoked into the ``collect()`` method

```python
def collect(self, project_dir: str | Path, file: str | None = None) -> None:
        """Return two fake dependencies for testing."""
        
        target = file or self.default_file

        if target not in self.dependency_files:
            raise ValueError(
                f"Unsupported file '{target}' for plugin '{self.name}'. "
                f"Supported: {', '.join(self.dependency_files)}"
            )

        project_dir = Path(project_dir)
        result = self._load_dependency_files(project_dir, target)

        if result is None:
            self.dependencies = []
            return

        data, lockfile_path = result
```

In [ ]:
# lets check the response object
# TODO: add a -y flag to confirm the scan
!cd depsight && uv run depsight poetry scan --project-dir /home/fixcfhu/repos/ValentinTwin1206/modern-python-devops-egineering/projects/proj8_depsighty/depsight/poetry-demo

#### 2.3.3 Collecting the dependencies

After we have loaded the data from the ``poetry.lock`` we need to return the dependencies as a ``Dependency`` array. Therefore, we meed to read the flat package list from the lockfile, figure out which package are top-level dependencies and which a pulled as transitively dependencies.

We start to grab the ``[[package]]`` list and we are preparing a few lookup tables that we fill while iterating over the packages:

```python
packages: list[dict] = data.get("package", []) # this is the list of packages

locked: dict[str, str] = {}           # name -> locked version,            e.g. {'colorama': '0.4.6'}
constraints: dict[str, str] = {}      # name -> constraint from a parent , e.g. {'python-dateutil': '>=2.6', 'tzdata': '>=2020.1'}
groups_map: dict[str, list[str]] = {} # name -> groups list,               e.g. {'colorama': ['dev']}         
referenced: set[str] = set()          # required packages,                 e.g. {'python-dateutil', 'tzdata'}
```

While walking through the packages we record the locked ``version`` and the ``groups`` of each package. The important part happens in the inner loop: every package listed inside another package's ``dependencies`` is added to the ``referenced`` set and its version constraint is remembered:

```python
for pkg in packages:
    name = pkg["name"]
    locked[name] = pkg["version"]
    groups_map[name] = pkg.get("groups", [])

    registry = pkg.get("source", {}).get("registry")
    if registry:
        sources[name] = registry

    for dep_name, specifier in pkg.get("dependencies", {}).items():
        referenced.add(dep_name)
        constraints.setdefault(dep_name, specifier)
```

Once we know which packages are referenced by others, the direct (root) dependencies are simply every locked package that is *never* referenced:

```python
root_dependencies = set(locked) - referenced
```

Finally, we turn each locked package into a ``Dependency`` object. A package is marked as ``is_transitive`` whenever it is *not* part of the root dependencies:

```python
for name in sorted(locked):
    deps.append(
        Dependency(
            name=name,
            version=locked[name],
            constraint=constraints.get(name),
            tool_name=self.name,
            registry=sources.get(name),
            file=lockfile_str,
            category=",".join(groups_map.get(name, [])),
            is_transitive=name not in root_dependencies,
        )
    )
```

In [ ]:
# now we need to test the implementation
!cd depsight && uv run depsight poetry scan --project-dir /home/fixcfhu/repos/ValentinTwin1206/modern-python-devops-egineering/projects/proj8_depsighty/depsight/poetry-demo

---

## 3. Testing our poetry plugin for depsight

#### 3.1 Providing the test fixtures

We are now at the point, to confirm our implementation by setting up suitable test scenarios for the plugin to work. We are going to setup ``pytest`` in order to fully test our ``depsight`` plugin.

To write reproducable and scalable tests, we are taking use of so called **fixtures**. A **fixture** is a reusable piece of setup code that prepares everything a test needs before it runs. 

In [ ]:
# copy the exmample poetry.lock into the fixtures folder.
!mkdir depsight/depsight-poetry-plugin/tests/fixtures && !cp depsight/poetry-demo/poetry.lock depsight/depsight-poetry-plugin/tests/fixtures

In order to avoid copying a fixture into every fike, we are going to provide a ``conftest.py`` which is a special kind of file, that ``pytest`` automatically discovers and which provides any fixture to all test files in the same directory—and even in subdirectories—without requiring any imports.

In [ ]:
# create a conftest.py file
!touch depsight/depsight-poetry-plugin/tests/conftest.py

```python
# our first fixture is the path to the poetry.lock file
@pytest.fixture()
def poetry_lock_file() -> Path:
    """Return the path to the `poetry.lock` fixture file."""
    return FIXTURES_DIR / "poetry.lock"


## our second fixture are the collected dependencies from the poetry.lock file
@pytest.fixture()
def collected_plugin(poetry_lock_file: Path) -> PoetryPlugin:
    """PoetryPlugin with dependencies already collected from the fixture."""
    plugin = PoetryPlugin()
    plugin.collect(project_dir=poetry_lock_file.parent)
    return plugin
```

In [ ]:
!cd depsight && uv run pytest

#### 3.2 Creating some more tests

We have prepared two reusable **fixture** - a ``poetry.lock`` and the collection of the dependencies. Both fixtures are discovered automatically by ``pytest``. We can simply invoke the **fixtures** by just referencing them as function input arguments

```python
 # reference the fixture in the method signature
 def test_collect_dependency_details(self, collected_plugin):
        plugin = collected_plugin
```
Let's finalize our test suite.

In [ ]:
!cd depsight && uv run pytest

In order to have a metric for our test setup, we can score the test coverage with the ``pytest-cov`` package

In [ ]:
# on root 
!cd depsight && uv add --dev pytest-cov

In [ ]:
coverage run -m pytest
coverage report -m

# 4. Summary

In this chapter, we explored the complete Python development workflow with uv—from creating projects and managing dependencies to organizing multi-package workspaces and building real functionality.

Along the way, we developed a ``depsight`` plugin for ``poetry``, learning plugin logic of ``python``. This separation keeps our application modular, predictable, and easy to extend.

Finally, we introduced automated testing with ``pytest``. Using **fixtures** allowed us to write reusable test setup, while the ``conftest.py`` helped us share that setup across multiple test files. Together with pytest-cov, we gained a simple but powerful feedback loop: write code, run tests, measure coverage, and continuously improve with confidence.

At this point, you have all the essential building blocks to develop, test, and maintain modern Python projects using a clean and reproducible workflow.